# 12/0.5 Dense Intersection: Creeping, Efficiency, and Safety Constraint

This report supersedes lower-density summaries for the target benchmark. The benchmark floor is `initial_vehicle_count=12`, `spawn_probability=0.5`, `duration=22`, and `negotiating_traffic=False`. No result below 12/0.5 is used as target evidence here.

## Efficiency Metrics

Creeping should not be judged only by collision rate. A policy can be safe by freezing, which is not negotiation. The added efficiency metrics are ego stopped time, ego delay proxy, traffic vehicle-seconds, traffic delay proxy, system delay proxy, and success per unit system delay.

## 20-Episode Crash-Tolerant Benchmark at 12/0.5

| maneuver | policy | episodes | success | collision | timeout | survival s | ego delay | traffic delay | system delay | creep-zone speed | high-speed zone | creep-speed | shield intervention |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| straight | learned_creeping | 20 | 60.0% | 40.0% | 0.0% | 13.44 | 5.34 | 28.35 | 33.69 | 3.96 | 0.8% | 99.0% | 0.0% |
| straight | fast_non_creeping | 20 | 35.0% | 65.0% | 0.0% | 6.36 | 0.00 | 5.49 | 5.49 | 8.98 | 100.0% | 0.0% | 0.0% |
| straight | learned_safety | 20 | 40.0% | 25.0% | 35.0% | 17.02 | 10.09 | 61.07 | 71.17 | 2.65 | 6.9% | 60.9% | 35.1% |
| left | learned_creeping | 20 | 50.0% | 50.0% | 0.0% | 12.13 | 4.90 | 34.13 | 39.03 | 3.82 | 4.5% | 90.5% | 0.0% |
| left | fast_non_creeping | 20 | 40.0% | 60.0% | 0.0% | 6.34 | 0.00 | 4.39 | 4.39 | 8.09 | 90.0% | 0.0% | 0.0% |
| left | learned_safety | 20 | 60.0% | 20.0% | 20.0% | 16.80 | 8.83 | 27.65 | 36.48 | 3.30 | 7.2% | 69.6% | 22.8% |
| straight | learned_safety_progress | 20 | 5.0% | 25.0% | 70.0% | 20.07 | 14.51 | 78.06 | 92.57 | 1.35 | 1.6% | 48.7% | 67.2% |
| left | learned_safety_progress | 20 | 5.0% | 10.0% | 85.0% | 21.47 | 16.39 | 98.89 | 115.28 | 1.05 | 0.5% | 38.8% | 69.0% |

## Raw Throughput Efficiency

This view treats every episode as a traffic system. It uses total system delay, traffic vehicle-seconds, and traffic delay per vehicle-second. It intentionally exposes the trap: a policy that crashes quickly can look fast on delay metrics.

| maneuver | policy | success/system-delay | safe crossing | unsafe or no crossing | system delay | traffic vehicle-s | traffic delay / vehicle-s |
| --- | --- | --- | --- | --- | --- | --- | --- |
| straight | learned_creeping | 0.0178 | 60.0% | 40.0% | 33.69 | 178.66 | 0.1587 |
| straight | fast_non_creeping | 0.0637 | 35.0% | 65.0% | 5.49 | 59.38 | 0.0925 |
| straight | learned_safety | 0.0056 | 40.0% | 60.0% | 71.17 | 257.57 | 0.2371 |
| left | learned_creeping | 0.0128 | 50.0% | 50.0% | 39.03 | 160.53 | 0.2126 |
| left | fast_non_creeping | 0.0911 | 40.0% | 60.0% | 4.39 | 46.27 | 0.0949 |
| left | learned_safety | 0.0164 | 60.0% | 40.0% | 36.48 | 183.73 | 0.1505 |
| straight | learned_safety_progress | 0.0005 | 5.0% | 95.0% | 92.57 | 324.26 | 0.2407 |
| left | learned_safety_progress | 0.0004 | 5.0% | 95.0% | 115.28 | 360.60 | 0.2742 |

## Negotiation-Efficiency Metrics

To answer whether creeping is efficient for negotiation, I added a second view. `safe-crossing efficiency` rewards success and penalizes collision/timeout, system delay, and ego stopped time. `conflict-discipline efficiency` additionally penalizes high-speed movement inside the conflict zone. `negotiation efficiency index` also requires actual creeping-speed behavior in the conflict zone. These are diagnostic scores, not physics constants; their job is to stop a crash-fast baseline from winning just because it fails quickly.

| maneuver | policy | safe-crossing efficiency | conflict-discipline efficiency | negotiation efficiency index | ego stopped s | high-speed zone | creep-speed |
| --- | --- | --- | --- | --- | --- | --- | --- |
| straight | learned_creeping | 0.2151 | 0.2134 | 0.2112 | 0.00 | 0.8% | 99.0% |
| straight | fast_non_creeping | 0.1104 | 0.0000 | 0.0000 | 0.00 | 100.0% | 0.0% |
| straight | learned_safety | 0.0476 | 0.0443 | 0.0270 | 4.69 | 6.9% | 60.9% |
| left | learned_creeping | 0.1404 | 0.1340 | 0.1213 | 0.00 | 4.5% | 90.5% |
| left | fast_non_creeping | 0.1471 | 0.0147 | 0.0000 | 0.00 | 90.0% | 0.0% |
| left | learned_safety | 0.1493 | 0.1386 | 0.0965 | 3.41 | 7.2% | 69.6% |
| straight | learned_safety_progress | 0.0006 | 0.0006 | 0.0003 | 7.38 | 1.6% | 48.7% |
| left | learned_safety_progress | 0.0005 | 0.0005 | 0.0002 | 9.87 | 0.5% | 38.8% |

## Interpretation

At 12/0.5, learned creeping is not raw-throughput efficient: it creates more total delay because it spends time negotiating. It is more efficient once negotiation is defined as safe crossing with conflict-zone discipline, especially compared with the fast baseline whose low delay mostly comes from crashing early. The safety-ellipse shield improves some risk measures but adds stops and timeouts, so it is not yet the right final controller. The progress-assist shield was worse in this 12/0.5 run because it mostly produced timeouts. The next required fix is training inside the shielded 12/0.5 dynamics with the interaction-focused reward.

## Forced-Zero Reset Audit

This audit forces ego speed to `0 m/s` immediately after reset and then steps the slowest action. If a collision still occurs, that reset is not preventable by an ego-only policy. The 12/0.5 seed slice below did not show such unavoidable stationary-ego collisions. So the remaining failures should be treated as policy/shield failures unless a larger 12/0.5 reset audit finds otherwise.

| benchmark | maneuver | tested resets | forced-zero collisions | collision rate | initial vehicles | spawn prob. | stationary steps |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 12/0.5 target | left | 100 | 0 | 0.0% | 12 | 0.5 | 3 |
| 12/0.5 target | straight | 100 | 0 | 0.0% | 12 | 0.5 | 3 |

## Shielded Fine-Tune Attempts

These runs fine-tune the calibrated creeping policy inside the safety-ellipse wrapper with the interaction-focused reward. The short v3 run used both maneuvers at 12/0.5 for 5k additional PPO steps. The result is not a solution: straight improves collision rate relative to the unshielded 12/0.5 learned policy but loses the creeping signature; left-turn training mostly turns collision risk into timeout/deadlock. This is the current evidence that the final controller needs a deeper shield-aware training run or a less brittle safety projection, not just a small fine-tune.

| attempt | maneuver | eval eps | success | collision | timeout | system delay | safe-crossing efficiency | high-speed zone | creep-speed | ego stopped s |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| safety fine-tune v1 | straight | 10 | 70.0% | 20.0% | 10.0% | 36.59 | 0.2398 | 69.2% | 17.5% | 1.56 |
| interaction-preserving v2 | straight | 10 | 70.0% | 10.0% | 20.0% | 51.01 | 0.1812 | 57.6% | 19.4% | 3.42 |
| shielded interaction v3 | straight | 20 | 65.0% | 15.0% | 20.0% | 49.39 | 0.1667 | 71.5% | 10.4% | 2.73 |
| shielded interaction v3 | left | 20 | 20.0% | 15.0% | 65.0% | 96.04 | 0.0086 | 1.0% | 46.9% | 8.57 |

## Rendered 12/0.5 Video Metrics

| policy | maneuver | videos | success | collision | ego delay | traffic delay | system delay |
| --- | --- | --- | --- | --- | --- | --- | --- |
| after_learned_safety | left | 10 | 40.0% | 20.0% | 13.08 | 74.57 | 87.65 |
| after_learned_safety | straight | 10 | 50.0% | 30.0% | 8.74 | 46.05 | 54.79 |
| before_learned_creeping | left | 10 | 50.0% | 50.0% | 5.64 | 29.72 | 35.36 |
| before_learned_creeping | straight | 10 | 50.0% | 50.0% | 5.27 | 24.28 | 29.55 |

## Video Root

`notebooks/results/route_videos/benchmark12_05_before_after_10eps`

## Before Constraint Videos

### before_learned_creeping / straight

`notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight`

<p><strong>straight episode 01</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep00.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep00.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep00.mp4</a></p>

<p><strong>straight episode 02</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep01.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep01.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep01.mp4</a></p>

<p><strong>straight episode 03</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep02.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep02.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep02.mp4</a></p>

<p><strong>straight episode 04</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep03.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep03.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep03.mp4</a></p>

<p><strong>straight episode 05</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep04.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep04.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep04.mp4</a></p>

<p><strong>straight episode 06</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep05.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep05.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep05.mp4</a></p>

<p><strong>straight episode 07</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep06.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep06.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep06.mp4</a></p>

<p><strong>straight episode 08</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep07.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep07.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep07.mp4</a></p>

<p><strong>straight episode 09</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep08.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep08.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep08.mp4</a></p>

<p><strong>straight episode 10</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep09.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep09.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/straight/benchmark12_05_before_learned_creeping_straight_straight_ep09.mp4</a></p>

### before_learned_creeping / left

`notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left`

<p><strong>left episode 01</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep00.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep00.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep00.mp4</a></p>

<p><strong>left episode 02</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep01.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep01.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep01.mp4</a></p>

<p><strong>left episode 03</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep02.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep02.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep02.mp4</a></p>

<p><strong>left episode 04</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep03.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep03.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep03.mp4</a></p>

<p><strong>left episode 05</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep04.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep04.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep04.mp4</a></p>

<p><strong>left episode 06</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep05.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep05.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep05.mp4</a></p>

<p><strong>left episode 07</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep06.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep06.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep06.mp4</a></p>

<p><strong>left episode 08</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep07.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep07.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep07.mp4</a></p>

<p><strong>left episode 09</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep08.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep08.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep08.mp4</a></p>

<p><strong>left episode 10</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep09.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep09.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/before_learned_creeping/left/benchmark12_05_before_learned_creeping_left_left_ep09.mp4</a></p>

## After Safety-Ellipse Constraint Videos

### after_learned_safety / straight

`notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight`

<p><strong>straight episode 01</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep00.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep00.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep00.mp4</a></p>

<p><strong>straight episode 02</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep01.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep01.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep01.mp4</a></p>

<p><strong>straight episode 03</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep02.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep02.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep02.mp4</a></p>

<p><strong>straight episode 04</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep03.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep03.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep03.mp4</a></p>

<p><strong>straight episode 05</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep04.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep04.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep04.mp4</a></p>

<p><strong>straight episode 06</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep05.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep05.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep05.mp4</a></p>

<p><strong>straight episode 07</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep06.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep06.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep06.mp4</a></p>

<p><strong>straight episode 08</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep07.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep07.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep07.mp4</a></p>

<p><strong>straight episode 09</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep08.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep08.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep08.mp4</a></p>

<p><strong>straight episode 10</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep09.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep09.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/straight/benchmark12_05_after_learned_safety_straight_straight_ep09.mp4</a></p>

### after_learned_safety / left

`notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left`

<p><strong>left episode 01</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep00.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep00.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep00.mp4</a></p>

<p><strong>left episode 02</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep01.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep01.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep01.mp4</a></p>

<p><strong>left episode 03</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep02.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep02.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep02.mp4</a></p>

<p><strong>left episode 04</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep03.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep03.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep03.mp4</a></p>

<p><strong>left episode 05</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep04.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep04.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep04.mp4</a></p>

<p><strong>left episode 06</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep05.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep05.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep05.mp4</a></p>

<p><strong>left episode 07</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep06.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep06.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep06.mp4</a></p>

<p><strong>left episode 08</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep07.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep07.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep07.mp4</a></p>

<p><strong>left episode 09</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep08.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep08.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep08.mp4</a></p>

<p><strong>left episode 10</strong><br><video controls width="520" src="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep09.mp4"></video><br><a href="results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep09.mp4">notebooks/results/route_videos/benchmark12_05_before_after_10eps/after_learned_safety/left/benchmark12_05_after_learned_safety_left_left_ep09.mp4</a></p>

## Current 100% Success Status

Not achieved. At the required 12/0.5 density, the current 20-episode benchmark is far below 100%. The current reset audit no longer gives us an easy impossibility excuse for this exact seed slice; the technical work is to make the learned policy cooperate with the safety constraint without freezing or abandoning creeping.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
REPO = ROOT.parent if ROOT.name == 'notebooks' else ROOT
result_dir = REPO / 'notebooks/results/intersectionRouteAttentionCreeping'
summary_paths = [
    result_dir / 'benchmark12_05_all_20eps_subprocess_summary.csv',
    result_dir / 'benchmark12_0_5_all_20eps_subprocess_summary.csv',
]
summary = pd.concat([pd.read_csv(path) for path in summary_paths if path.exists()], ignore_index=True)
summary = summary.drop_duplicates(['maneuver', 'policy_family'], keep='last').reset_index(drop=True)
assert (summary['initial_vehicle_count'] == 12).all()
assert (summary['spawn_probability'] == 0.5).all()
assert (summary['negotiating_traffic'].astype(str).str.lower() == 'false').all()
summary